In [ ]:
import numpy as np
from case_studies import * 

# Utility to project initial point onto feasible set
def project_initial_point(x0, A, b):
    """Project x0 onto the set {x | Ax + b = 0}."""
    return x0 - A.T @ np.linalg.inv(A @ A.T) @ (A @ x0 + b)

def backtracking_line_search(f, df, x, p, alpha=1.0, rho=0.5, c1=1e-4):
    while f(x + alpha * p) > f(x) + c1 * alpha * np.dot(df(x), p):
        alpha *= rho
    return alpha

def solve_linear_system(B, A, grad_f):
    """Solve the KKT system [B A^T; A 0][p; lambda] = [-grad_f; 0]."""
    m, n = A.shape
    zero_block = np.zeros((m, m))
    lhs = np.block([[B, A.T], [A, zero_block]])
    rhs = np.concatenate([-grad_f, np.zeros(m)])
    try:
        solution = np.linalg.solve(lhs, rhs)
        p_k = solution[:n]
        lambda_k = solution[n:]
        return p_k, lambda_k
    except np.linalg.LinAlgError:
        print("Warning: Singular matrix in KKT system")
        return None, None

# Constrained Newton's Method
def constrained_newton(f, df, Hf, A, b, x0, tol=1e-9, max_iter=1000, alpha_init=1.0, rho=0.5, c1=1e-4):
    """Newton's method with linear equality constraints Ax + b = 0."""
    x_k = project_initial_point(x0, A, b)  # Ensure initial feasibility
    xs = [x_k.copy()]
    grad_norms = []

    for k in range(max_iter):
        grad = df(x_k)
        H = Hf(x_k)
        
        # Check Hessian positive definiteness
        eigvals = np.linalg.eigvalsh(H)
        if eigvals.min() > 0:
            B = H
        else:
            eigvals, eigvecs = np.linalg.eigh(H)
            B = sum(np.abs(l) * np.outer(v, v) for l, v in zip(eigvals, eigvecs.T))

        # Solve KKT system
        p_k, lambda_k = solve_linear_system(B, A, grad)

        # Check constraint satisfaction
        constraint_error = np.linalg.norm(A @ x_k + b)
        if constraint_error > 1e-10:
            print(f"Iteration {k}: Constraint violation = {constraint_error}")
        
        # Stopping criterion:
        I = np.eye(A.shape[1])
        grad_norm = np.linalg.norm((I-A.T @ np.linalg.inv(A @ A.T) @ A) @ grad)
        grad_norms.append(grad_norm)
        if grad_norm < tol:
            break

        alpha_k = backtracking_line_search(f, df, x_k, p_k, alpha_init, rho, c1)
        x_k = x_k + alpha_k * p_k
        xs.append(x_k.copy())

    return np.array(xs), np.array(grad_norms)

# Constrained Steepest Descent
def constrained_steepest_descent(f, df, A, b, x0, tol=1e-6, max_iter=1000, c1=1e-4, rho=0.5):
    """Steepest Descent with linear equality constraints Ax + b = 0."""
    x_k = project_initial_point(x0, A, b)  # Ensure initial feasibility
    xs = [x_k.copy()]
    grad_norms = []
    m, n = A.shape
    M = np.eye(n) - A.T @ np.linalg.inv(A @ A.T) @ A  # Projection matrix
    beta_k = 1.0

    for k in range(max_iter):
        grad = df(x_k)
        
        # Check constraint satisfaction
        constraint_error = np.linalg.norm(A @ x_k + b)
        if constraint_error > 1e-10:
            print(f"Iteration {k}: Constraint violation = {constraint_error}")

        # Projected gradient
        proj_grad = M @ grad
        
        # Stopping criterion:
        I = np.eye(A.shape[1])
        grad_norm = np.linalg.norm((I-A.T @ np.linalg.inv(A @ A.T) @ A) @ grad)
        grad_norms.append(grad_norm)
        if grad_norm < tol:
            break

        p_k = -proj_grad
        alpha_k = backtracking_line_search(f, df, x_k, p_k, beta_k, rho, c1)
        x_k = x_k + alpha_k * p_k
        xs.append(x_k.copy())
        beta_k = alpha_k / rho

    return np.array(xs), np.array(grad_norms)

In [ ]:
import matplotlib.pyplot as plt
def test_optimizers_on_function_distance(functions, optimizers, A, b, dimension, alpha_init=1.0, rho=0.5, c1=1e-4, c2=0.9, epsilon=1e-9, max_iter=1000, num_runs=5):
    """
    Test optimizers on a given set of functions and plot ||x_k - x*|| vs Iterations.
    Runs multiple times with different starting points and plots each individually.

    Parameters:
    - functions: List of tuples (function, gradient, Hessian)
    - optimizers: List of optimizers (functions)
    - A: Constraint matrix
    - b: Constraint vector
    - dimension: Dimension of the problem (default is 20)
    - epsilon: Tolerance for stopping condition (default is 1e-9)
    - max_iter: Maximum number of iterations (default is 1000)
    - num_runs: Number of runs with different initializations (default is 5)
    """
    colors = ['b', 'r', 'g', 'orange', 'purple']  # Assign unique colors for each run

    for f, df, Hf in functions:
        plt.figure(figsize=(6, 4))
        plt.title(f"{f.__name__} (d={dimension}, ε={epsilon:.0e}, α_init={alpha_init}, ρ={rho}, c1={c1})")

        for opt_idx, opt in enumerate(optimizers):
            color = colors[opt_idx % len(colors)]  

            for run in range(num_runs):
                x0 = np.random.uniform(-3, 3, size=dimension)

                x0 = project_initial_point(x0, A, b)

                if opt == constrained_newton:
                    xs,_ = opt(f, df, Hf, A, b, x0, tol=epsilon, max_iter=max_iter, alpha_init=alpha_init, rho=rho, c1=c1)
                elif opt == constrained_steepest_descent:
                    xs,_ = opt(f, df, A, b, x0, tol=epsilon, max_iter=max_iter, c1=c1, rho=rho)

                optimal = x_opt(f.__name__, dimension)
                distances = [np.linalg.norm(x - optimal) for x in xs]

                label = f"{opt.__name__}" if run == 0 else ""
                plt.plot(range(len(distances)), distances, linestyle='--', color=color, alpha=0.8, label=label)

        plt.xlabel("Iterations")
        plt.ylabel(r"$||x_k - x^*||$")
        plt.yscale("log") 
        plt.grid()
        plt.legend()
        plt.show()

In [ ]:
m,n = 3,5 # m should <= n
A =  np.random.uniform(-3,3, (m, n))  # Constraint matrix (m linear constraints)
b = np.random.uniform(-3,3, m)  # Constraint vector

functions = [(f1, df1, Hf1), (f3, df3, Hf3), (f4, df4, Hf5), (f5, df5, Hf5)]

optimizers = [constrained_newton, constrained_steepest_descent]

test_optimizers_on_function_distance(functions, optimizers, A, b, dimension=n, alpha_init=1.0, rho=0.5, c1=1e-4, c2=0.9, epsilon=1e-9, max_iter=1000, num_runs=1)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def test_optimizers_on_function_gradnorm(functions, optimizers, A, b, dimension, alpha_init=1.0, rho=0.5, c1=1e-4, epsilon=1e-9, max_iter=1000, num_runs=5):
    """
    Test optimizers on a given set of functions and plot gradient norm vs Iterations.
    Runs multiple times with different starting points and plots each individually.

    Parameters:
    - functions: List of tuples (function, gradient, Hessian)
    - optimizers: List of optimizers (functions)
    - A: Constraint matrix
    - b: Constraint vector
    - dimension: Dimension of the problem (default is 20)
    - epsilon: Tolerance for stopping condition (default is 1e-9)
    - max_iter: Maximum number of iterations (default is 1000)
    - num_runs: Number of runs with different initializations (default is 5)
    """
    colors = ['b', 'r', 'g', 'orange', 'purple']  # Assign unique colors for each optimizer

    for f, df, Hf in functions:
        plt.figure(figsize=(6, 4))
        plt.title(f"{f.__name__} (d={dimension}, ε={epsilon:.0e}, α_init={alpha_init}, ρ={rho}, c1={c1})")

        for opt_idx, opt in enumerate(optimizers):
            color = colors[opt_idx % len(colors)]  # Unique color per optimizer

            for run in range(num_runs):
                x0 = np.random.uniform(-3, 3, size=dimension)

                if opt.__name__ == 'constrained_newton':
                    xs, grad_norms = opt(f, df, Hf, A, b, x0, tol=epsilon, max_iter=max_iter, alpha_init=alpha_init, rho=rho, c1=c1)
                elif opt.__name__ == 'constrained_steepest_descent':
                    xs, grad_norms = opt(f, df, A, b, x0, tol=epsilon, max_iter=max_iter, c1=c1, rho=rho)
                else:
                    raise ValueError(f"Unknown optimizer: {opt.__name__}")

                # Use grad_norms directly (already computed by the optimizers)
                label = f"{opt.__name__}" if run == 0 else ""  # Label only first run per optimizer
                plt.plot(range(len(grad_norms)), grad_norms, linestyle='--', color=color, alpha=0.8, label=label)

        plt.xlabel("Iterations")
        plt.ylabel(r"$||M\nabla f(x_k)||$")  # Adjusted for constrained context
        plt.yscale("log")  # Log scale for better visualization
        plt.grid()
        plt.legend()
        plt.show()

In [ ]:
m,n = 3,5 # m should <= n


A =  np.random.uniform(-3,3, (m, n))  # Constraint matrix (m linear constraints)
b = np.random.uniform(-3,3, m)  # Constraint vector

functions = [(f1, df1, Hf1), (f3, df3, Hf3), (f4, df4, Hf5), (f5, df5, Hf5)]

optimizers = [constrained_newton, constrained_steepest_descent]

test_optimizers_on_function_gradnorm(functions, optimizers, A, b, dimension=n, alpha_init=1.0, rho=0.5, c1=1e-4, epsilon=1e-9, max_iter=1000, num_runs=1)

In [ ]:
m,n = 1,2 # m should <= n

A =  np.random.uniform(-3,3, (m, n))  # Constraint matrix (m linear constraints)
b = np.random.uniform(-3,3, m)  # Constraint vector

functions = [ (f2, df2, Hf2)]

optimizers = [constrained_newton, constrained_steepest_descent]

test_optimizers_on_function_gradnorm(functions, optimizers, A, b, dimension=n, alpha_init=1.0, rho=0.5, c1=1e-4, epsilon=1e-9, max_iter=1000, num_runs=1)